# 2. 데이터 합성

- 합성 데이터(Synthesis Data) 생성 과정을 학습
- ‘영화 추천’ 테스크를 위한 새로운 학습 데이터를 LLM을 통해 직접 만들어 볼 것

## 2-1. 데이터 생성을 위한 Prompt Engineering

### 2-1-1. Synthesis이란?

- 실제 수집한 데이터가 아닌, LLM(거대 언어 모델)과 같은 모델을 통해 인공적으로 만들어낸 데이터를 의미
- 이는 단순히 기존 데이터를 약간 변형하여 양을 늘리는 데이터 증강(Data Augmentation)과는 다른 개념. 데이터 증강이 원본 이미지의 밝기를 조절하거나 문장의 단어 순서를 바꾸는 것이라면, 데이터 합성은 세상에 없던 완전히 새로운 데이터 쌍(예: 새로운 질문과 그에 대한 답변)을 창조해내는 방식
- 데이터 합성은 학습 데이터가 부족하거나, 실제 데이터를 수집하기 어려운 민감한 정보(개인정보, 의료정보 등)를 다뤄야 할 때 매우 유용한 해결책이 됨

### 2-1-2. 기본 구조

- 효과적인 프롬프트를 작성하기 위해서는 LLM에게 명확한 가이드라인을 제시해야 함. 좋은 프롬프트는 다음과 같은 기본 구조를 가집

1. **역할(role):**: AI에게 특정 역할을 부여하여 답변의 전문성과 톤앤매너를 설정. 예를 들어, "당신은 세계적인 영화 평론가입니다." 와 같이 역할을 지정하면, 더 전문적인 시각에서 답변을 생성하게 됨
2. **목표(task):**: AI가 수행해야 할 작업을 구체적이고 명확하게 지시. "사용자의 취향에 맞는 영화를 추천해 주세요."처럼 목표가 명확할수록 결과물의 품질이 높아집
3. **조건(constraints):**: 답변의 형식, 스타일, 내용 등에 대한 제약 조건을 설정. "영화는 반드시 한 편만 추천하고, 추천 이유는 세 문장 이내로 작성하세요." 와 같이 조건을 구체화하면, 원하는 결과물을 일관되게 얻을 수 있음

### 2-1-3. 데이터 합성 프로프트의 핵심

- 양질의 합성 데이터를 만들기 위해서는 결과물의 **다양성**과 **일관성**을 모두 확보해야 함

1. **다양성 확보(temperature, top_p)**
  - 모델 답변의 무작위성을 조절하여 매번 새롭고 다채로운 데이터를 생성. 이 무작위성은 모델이 다음 단어를 선택할 때 확률 분포를 어떻게 사용하느냐에 따라 결정됨
    - 모델 답변의 무작위 성을 조절
    - `temperature`: 확률 분포의 모양을 조절하는 매개변수
        - 값이 **0에 가까울수록(온도가 낮을수록)**, 모델은 확률이 가장 높은 단어를 선택하려는 경향이 강해져 일관되고 결정적인 답변을 생성. 항상 동일한 질문에 거의 동일한 답변이 나오게 됨
        - 값이 **1에 가까울수록(온도가 높을수록)**, 확률이 낮은 단어들도 선택될 가능성이 커져 창의적이고 다양한 답변이 생성. 같은 질문에도 매번 다른 스타일의 답변을 기대할 수 있음
    - `top_p`: **핵심 샘플링(Nucleus Sampling)**이라고도 불리며, 후보 단어의 범위를 동적으로 조절
        - 모델이 다음 단어를 예측할 때, 확률이 높은 순서대로 단어들의 누적 확률을 계산. 이 누적 확률이 우리가 설정한 `top_p` 값에 도달하는 순간, 그 단어들까지만 후보군으로 사용
        - 예를 들어, `top_p=0.5`로 설정하면, 상위 단어들의 누적 확률이 50%가 될 때까지의 단어들 중에서만 다음 단어를 선택. 이를 통해 문맥에 따라 후보군의 크기를 유연하게 조절하여, 너무 엉뚱한 단어는 배제하면서도 적절한 수준의 다양성을 확보할 수 있음
2. **일관성 확보**
    - 생성된 데이터가 `{"movie_name": ..., "year": ..., "reason": ...}`과 같이 일관된 구조를 갖도록 프롬프트 내에 **JSON 형식**을 명시해야 함
    - 이는 사람이 직접 파싱(parsing)하는 수고를 덜어줄 뿐만 아니라, 후속 처리(데이터베이스 저장, 모델 학습 등)를 자동화하는 데 필수적
    - Chapter1에서 다루었던 `response_format` 기능을 활용하면, 모델이 반드시 지정된 JSON 형식에 맞춰 답변하도록 강제할 수 있어 데이터의 일관성을 효과적으로 확보할 수 있음

## 2-2. 실습 코드

### 2-2-1. 환경 설정

In [ ]:
# 구글 드라이브를 코랩 환경에 마운트합니다.
# 이를 통해 드라이브에 저장된 파일(.env 등)에 접근할 수 있습니다.
from google.colab import drive
drive.mount('/content/drive')

# API 키 파일이 저장된 기본 경로를 설정합니다.
base_path = '/content/drive/MyDrive/Colab Notebooks/AI/08_Data_Synthesis/'

In [ ]:
# .env 파일에서 환경 변수를 로드하기 위한 라이브러리.
from dotenv import load_dotenv
# 운영체제의 환경 변수를 가져오기 위한 함수.
from os import getenv
from pprint import pprint

# .env 파일을 로드하여 환경 변수를 설정.
load_dotenv(base_path + ".env")

# getenv 함수를 사용해 "UPSTAGE_API_KEY"라는 이름의 환경 변수 값을 가져옴.
UPSTAGE_API_KEY = getenv("UPSTAGE_API_KEY")

# API 키가 성공적으로 로드되었는지 확인하고 메시지를 출력.
if UPSTAGE_API_KEY:
    print("Success API Key Setting!")
else:
    print(f"ERROR: Failed to load UPSTAGE_API_KEY from {base_path}")
    

### 2-2-2. 데이터 생성

In [ ]:
# Upstage API와 통신하기 위해 openai 라이브러리를 임포트합니다.
from openai import OpenAI
# LLM의 응답(JSON 형식의 문자열)을 파이썬 딕셔너리로 변환하기 위해 json 라이브러리를 임포트합니다.
import json

# 시스템 프롬프트: AI 모델의 역할과 기본 행동 지침을 정의합니다.
SYSTEM_PROMPT = """
당신은 세상의 모든 영화를 꿰뚫고 있는 영화 전문가 '시네마스터'입니다.
사용자의 요청에 맞춰 영화를 추천하는 역할을 맡고 있습니다. 영화는 반드시 하나만 추천합니다.
"""

# 추가 규칙 프롬프트: AI 모델의 말투나 답변 스타일 등 세부 규칙을 정의합니다.
RULE = """
친구가 소개 해주는 듯 부드럽고 친근한 말투로 답변합니다.
특히, recommended_reason 항목에서는 친구가 엄청 호들갑 떨듯이 설명해 주세요.
"""

# 응답 형식 프롬프트: LLM의 답변을 구조화된 JSON 형식으로 강제하기 위한 설정입니다.
response_format = {
    "type": "json_schema",      # 응답 형식을 JSON 스키마로 지정합니다.
    "json_schema": {
        "name": "영화 추천",      # 스키마의 이름을 지정합니다.
        "strict": True,          # 엄격 모드: 스키마에 맞지 않는 응답이 오면 오류를 발생시킵니다.
        "schema": {
            "type": "object",    # 응답의 최상위 타입이 객체(딕셔너리)임을 지정합니다.
            "properties": {      # 객체에 포함될 속성들을 정의합니다.
                "movie_name": {"type": "string"},
                "year": {"type": "integer"},
                "reason": {"type": "string"},
                "description": {"type": "string", "description": "영화에 대한 설명"},
                "recommended_reason": {"type": "string", "description": "이 영화를 추천하는 추가 이유"}
            },
            # 필수적으로 포함되어야 할 속성들을 지정합니다.
            "required": ["movie_name", "year", "reason", "description", "recommended_reason"]
        }
    }
}

# Upstage API 클라이언트를 생성합니다.
client = OpenAI(
    api_key=UPSTAGE_API_KEY,
    base_url="https://api.upstage.ai/v1"
)

# client.chat.completions.create 메서드를 호출하여 LLM에 요청을 보냅니다.
response = client.chat.completions.create(
    model="solar-pro2",
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "system",
            "content": RULE
        },
        {
            "role": "user",
            "content": "공포 영화를 추천해줘"
        }
    ],
    response_format=response_format,  # 위에서 정의한 JSON 응답 형식을 적용합니다.
    temperature=0.5,                  # 다양성을 위해 temperature를 0.5로 설정합니다.
    max_tokens=1000,                  # 응답의 최대 길이를 1000 토큰으로 제한합니다.
    top_p=1.0,                        # top_p를 1.0으로 설정하여 모든 단어를 후보로 고려합니다.
    n=1,                              # 1개의 응답만 생성합니다.
    frequency_penalty=0.0,            # 특정 단어의 반복을 억제하지 않습니다.
    presence_penalty=0.0              # 새로운 주제의 등장을 장려하지 않습니다.
)

# 응답 결과에서 실제 텍스트 내용만 추출합니다.
output = response.choices[0].message.content
print(output)

## 2-3. 생성 데이터 평가 (LLM as a Judge)

### 2-3-1. LLM as a Judge란?

- 사람이 직접 수많은 합성 데이터의 품질을 하나하나 검수하는 것은 시간과 비용이 많이 드는 비효율적인 작업
- **LLM as a Judge**는 이러한 검수 과정을 자동화하기 위해, **다른 LLM을 '평가자'로 활용**하는 기법
- 이 방법의 가장 큰 장점은 단순히 "좋다/나쁘다"와 같은 정량적인 평가를 넘어, **"왜 그렇게 평가했는지"에 대한 이유까지 생성**하여 데이터의 어떤 부분을 개선해야 할지 구체적인 피드백을 얻을 수 있다는 점

### 2-3-2. 생성 평가의 핵심

1. **평가 기준 설정**
    - 평가의 목적을 명확히 해야 함. 이번 실습에서는 생성된 영화 정보의 사실 여부를 검증하는 것이 아니라, 사전에 우리가 지시했던 `RULE`**(친근한 말투, 호들갑 떠는 설명 등)을 얼마나 잘 이행했는지를 평가**하는 데 초점을 맞춤
2. **일관성 확보 (temperature=0)**
    - `평가자` 역할을 맡은 LLM은 창의적이거나 다양한 답변을 할 필요가 없음. 오히려 동일한 입력에 대해 항상 **일관되고 객관적인 평가**를 내려야 신뢰할 수 있음
    - 따라서 평가자 LLM을 호출할 때는 `temperature` **값을 항상 0으로 설정**하는 것이 권장됨
3. **체계적인 평가 프롬프트 설계**
    - 평가자에게 필요한 모든 정보를 명확하게 제공해야 함
    - **입력:** 평가자 LLM에게 **[우리가 내렸던 지시사항], [생성 모델의 답변]**, 그리고 **[평가 기준]**을 모두 명시적으로 전달해야 함
    - **출력:** 평가자 역시 답변을 `score`(점수)와 `comment`(평가 이유)를 포함하는 **JSON 형식**으로 생성하도록 엄격하게 지시하여, 평가 결과를 쉽게 파싱하고 활용할 수 있도록 해야 함

### 2-3-3. 평가 실습 코드

In [ ]:
# '평가자' 역할을 수행할 LLM에게 제공할 시스템 프롬프트입니다.
JUDGE_SYSTEM_PROMPT = """
당신의 역할은 모델 답변 자동 평가자입니다.

1. 입력 형식
    - 입력 프롬프트: [instruction]
    - 모델 답변: [output]
    - 평가 기준: [criteria]

2. 작업 지시
    - [instruction]에 따른 모델 결과물인 [output]을 평가합니다.
    - [output]은 [criteria]를 충족하는지 평가합니다.

3. 채점 원칙 (각 기준별 1–5점, 정수만)
    - 5점 (탁월): 기준을 완전히 충족. 오류·누락 없음. 구체적이고 실행가능.
    - 4점 (우수): 대체로 충족. 사소한 흠만 있음(정확성·구체성·형식 등에서 경미한 누락).
    - 3점 (보통): 핵심은 맞지만 눈에 띄는 약점 존재(누락, 모호함, 근거 부족 등).
    - 2점 (미흡): 중요한 요구를 여러 곳에서 놓침 또는 오류/비논리 다수.
    - 1점 (부적합): 전반적으로 요청과 어긋남, 의미있는 도움/근거 없음, 안전·정책 위반 가능성.

4. 출력 형식 (엄격 준수)
    - "score"는 1–5점의 정수로 평가한다.
    - "comment"는 한국어 1–3문장으로 평가한다. 구체적이고 실행 가능하게 작성한다.
    - 출력 형식은 JSON 형식인 response_format을 준수한다.
"""

# 평가자 LLM의 응답 형식을 JSON으로 강제하기 위한 설정입니다.
judge_response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "영화 추천 평가자",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "score": {"type": "integer"},
                "comment": {"type": "string", "description": "모델의 답변에 대한 평가 주석"}
            },
            "required": ["score", "comment"]
        }
    }
}

# 평가자 LLM에게 전달할 사용자 프롬프트 템플릿입니다.
USER_PROMPT = """
입력 프롬프트: {instruction}
모델 답변: {output}
평가 기준: {criteria}
"""

# 평가에 사용할 변수들을 정의합니다.
instruction = "공포 영화를 추천해줘"
# output 변수는 이전 데이터 생성 단계에서 얻은 결과물을 그대로 사용합니다.
# output = output

# 평가 기준은 생성 모델에게 전달했던 모든 지시사항(시스템 프롬프트, 규칙, 응답 형식)을 조합하여 만듭니다.
criteria = SYSTEM_PROMPT + RULE + str(response_format)

# 평가자 LLM에게 요청을 보냅니다.
response = client.chat.completions.create(
    model="solar-pro2",
    messages=[
        {
            "role": "system",
            "content": JUDGE_SYSTEM_PROMPT
        },
        {
            "role": "user",
            # .format()을 사용해 템플릿에 실제 변수 값들을 채워 넣습니다.
            "content": USER_PROMPT.format(instruction=instruction, output=output, criteria=criteria)
        }
    ],
    temperature=0,  # 평가의 일관성을 위해 temperature를 0으로 설정합니다.
    response_format=judge_response_format
)

# 평가 결과를 JSON 문자열에서 파이썬 딕셔너리로 변환합니다.
judge_output = json.loads(response.choices[0].message.content)

# 최종 평가 결과를 확인합니다.
pprint(judge_output)

## [참고] 데이터 증강 (Data Augmentation)

- 이번 실습에서는 LLM을 활용한 **데이터 합성(Synthesis)**, 즉 완전히 **새로운 데이터 쌍**을 생성하는 방법에 초점을 맞추었음.

- 이와 관련된 중요한 기법으로 `데이터 증강(Data Augmentation)`이 있음.

- 데이터 증강은 **기존에 보유한 데이터를 기반으로 변형** (예: 이미지 회전, 텍스트 동의어 교체)을 가해 데이터 양을 늘리는 방식. 이는 모델의 강건성(robustness)을 높이고 과적합을 방지하는 데 도움을 줌.

- 데이터 합성이 '무(無)'에서 '유(有)'를 창조하는 것이라면, 데이터 증강은 '유(有)'에서 '또 다른 유(有)'를 만들어내는 과정으로 볼 수 있음.

- 데이터 증강 기법의 **실질적인 효과**(예: 모델 성능 향상 기여도)를 검증하기 위해서는, 증강된 데이터를 활용하여 **모델을 재학습하고 성능 변화를 면밀히 비교·분석하는 추가적인 과정**이 필요함.

- 본 실습은 **LLM을 활용한 새로운 데이터 생성 및 파운데이션 모델 활용**에 중점을 두고 있으므로, 데이터 증강의 심층적인 효과 검증까지 다루기에는 범위가 다름.

- 이번 실습에서는 다루지 않았지만, 모델의 성능을 높이기 위해 데이터 합성과 함께 고려해볼 수 있는 중요한 기법 중 하나.